# Data Challenge 11 — Evaluating MLR & Fixing Multicollinearity (HVFHV Trips)


**Format:** Instructor Guidance → You Do (Students) → We Share (Reflection)

**Goal:** Build an MLR, evaluate it with a **train–test split**, diagnose multicollinearity with **corr** and **VIF** on the **training set**, fix issues (drop/choose features), and report **test MAE/RMSE** + **coefficient interpretations**.

**Data:** July 1, 2023 - July 15, 2023 For Hire Vehicle Data in NYC

[July For Hire Vehicles Data](https://data.cityofnewyork.us/Transportation/2023-High-Volume-FHV-Trip-Data/u253-aew4/about_data)


## Instructor Guidance

**Hint: Use the Lecture Deck, Canvas Reading, and Docs to help you with the code**

Use this guide live; students implement below.

**Docs (quick links):**
- Train/Test Split — scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- OLS — statsmodels: https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html
- OLS Results (rsquared_adj, pvalues, resid, etc.): https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.html
- VIF — statsmodels: https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.variance_inflation_factor.html
- Corr — pandas: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html

### Pseudocode Plan (Evaluation + Multicollinearity)
1) **Load CSV** → preview shape/columns; (optional) filter to **July**.
2) **Pick Y** (`base_passenger_fare`) and **candidate X’s** (e.g., `trip_miles`, `trip_time_minutes`, `tolls`, `tips` if present).
3) **Light prep** → derive `trip_time_minutes` from `trip_time` (seconds) if present; coerce only used cols to numeric; drop NA rows.
4) **Split** → `X_train, X_test, y_train, y_test` (80/20, fixed `random_state`).
5) **Diagnose on TRAIN**:
   - **Correlation matrix** (|r| > 0.7 = red flag).
   - **VIF** for each predictor (1–5 ok; >5–10+ = concerning).
6) **Fix** → drop/choose among highly correlated predictors (business logic).
7) **Fit on TRAIN only** → OLS with intercept.
8) **Predict on TEST** → compute **MAE/RMSE** (units of Y).
9) **Interpret** → unit-based coefficient sentences **holding others constant**; note any changes after fixing collinearity.
10) **Report** → table of (features kept, Adj R², MAE, RMSE) + 1-line stakeholder takeaway.


## You Do — Student Section
Work in pairs. Comment your choices briefly. Keep code simple—only coerce the columns you use.

### Step 0 — Setup & Imports

In [6]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

### Step 1 — Load CSV & Preview
- Point to your For Hire Vehicle Data 
- Print **shape** and **columns**.

**Hint: You may have to drop missing values and do a force coercion to make sure the variables stay numeric (other coding assignments may help)**

In [7]:
df = pd.read_csv('/Users/Marcy_Student/Downloads/FHV_072023 copy.csv')
display(df.info())
display(df.head())

/var/folders/7n/rj4gbkk13_1f9n1qf54j12gh0000gp/T/ipykernel_65309/2873762553.py:1: DtypeWarning: Columns (11,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/Marcy_Student/Downloads/FHV_072023 copy.csv')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8324591 entries, 0 to 8324590
Data columns (total 24 columns):
 #   Column                Dtype  
---  ------                -----  
 0   hvfhs_license_num     object 
 1   dispatching_base_num  object 
 2   originating_base_num  object 
 3   request_datetime      object 
 4   on_scene_datetime     object 
 5   pickup_datetime       object 
 6   dropoff_datetime      object 
 7   PULocationID          int64  
 8   DOLocationID          int64  
 9   trip_miles            float64
 10  trip_time             object 
 11  base_passenger_fare   object 
 12  tolls                 float64
 13  bcf                   float64
 14  sales_tax             float64
 15  congestion_surcharge  float64
 16  airport_fee           float64
 17  tips                  float64
 18  driver_pay            object 
 19  shared_request_flag   object 
 20  shared_match_flag     object 
 21  access_a_ride_flag    object 
 22  wav_request_flag      object 
 23  wav_mat

None

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0005,B03406,NaN,07/01/2023 05:34:30 PM,NaN,07/01/2023 05:37:48 PM,07/01/2023 05:44:45 PM,158,68,1.266,...,1.35,2.75,0.0,2.00,5.57,N,N,N,N,False
1,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:36:53 PM,07/01/2023 05:37:15 PM,07/01/2023 05:55:15 PM,162,234,2.350,...,1.52,2.75,0.0,3.28,13.38,N,N,NaN,N,False
2,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:35:17 PM,07/01/2023 05:35:52 PM,07/01/2023 05:44:27 PM,161,163,0.810,...,0.49,2.75,0.0,0.00,5.95,N,N,NaN,N,False
3,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:37:39 PM,07/01/2023 05:39:35 PM,07/01/2023 06:23:02 PM,122,229,15.470,...,5.17,2.75,0.0,0.00,54.46,N,N,NaN,N,True
4,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:36:06 PM,07/01/2023 05:36:39 PM,07/01/2023 05:45:06 PM,67,14,1.520,...,0.85,0.00,0.0,3.00,7.01,N,N,NaN,N,False


In [8]:
# Cleaning
num_cols = ['base_passenger_fare', 'trip_miles', 'trip_time', 'tolls', 'tips']

for c in num_cols:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.strip().str.replace(r'[^0-9.+\-eE]', '', regex=True),
        errors='coerce'
    )

df = df[(df['base_passenger_fare'] > 0) & (df['trip_miles'] > 0) & (df['trip_time'] > 0)]
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=num_cols)

def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.10)
    Q3 = df[column].quantile(0.90)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

for col in ['base_passenger_fare', 'trip_miles', 'trip_time']:
    df = remove_outliers_iqr(df, col)


### Step 2 —  Choose Target **Y** and Candidate Predictors

- Suggested **Y**: `base_passenger_fare` (USD).
- Start with **distance** and **time**; optionally add **flags** if present.
- Derive `trip_time_minutes` from `trip_time` (seconds) if available.

In [ ]:
df['trip_time_minutes'] = df['trip_time'] / 60

Y = df['base_passenger_fare']
X = df[['trip_miles', 'trip_time_minutes', 'tolls', 'tips']]

### Step 3 — Train–Test Split

- Use a fixed `random_state` for reproducibility.
- **All diagnostics below must be done on TRAIN only.**

In [10]:
np.random.seed(42)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

### Step 4 — Diagnose Multicollinearity on **TRAIN** — Correlation Matrix
- Flag any |r| > 0.70 as a potential problem.


In [ ]:
corr_matrix = X_train.corr()
print(corr_matrix)

# W .any() function
if any((corr_matrix.abs() > 0.7).any()):
    print("Potential multicollinearity detected!")
else:
    print("No strong correlations detected among predictors.")

                   trip_miles  trip_time_minutes     tolls      tips
trip_miles           1.000000           0.806937  0.471286  0.246612
trip_time_minutes    0.806937           1.000000  0.364870  0.241719
tolls                0.471286           0.364870  1.000000  0.183295
tips                 0.246612           0.241719  0.183295  1.000000
Potential multicollinearity detected!


### Step 5 — Diagnose Multicollinearity on **TRAIN** — VIF
- 1–5 normal; >5–10+ concerning.

In [17]:
X_vif = sm.add_constant(X_train)
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)


             feature       VIF
0              const  3.638553
1         trip_miles  3.208634
2  trip_time_minutes  2.886274
3              tolls  1.295142
4               tips  1.077780


### Step 6 — Fix High VIF (if needed)

- If two predictors are highly correlated, **drop/choose** using business logic (e.g., keep the more actionable one).
- Recompute VIF to confirm improvement.

In [ ]:
X_train_reduced = X_train.drop(columns=['trip_time_minutes'])
X_test_reduced = X_test.drop(columns=['trip_time_minutes'])

# Recalculating VIF
X_vif_reduced = sm.add_constant(X_train_reduced)
vif_data_reduced = pd.DataFrame()
vif_data_reduced['feature'] = X_vif_reduced.columns
vif_data_reduced['VIF'] = [variance_inflation_factor(X_vif_reduced.values, i) for i in range(len(X_vif_reduced.columns))]
print(vif_data_reduced)

      feature       VIF
0       const  2.172618
1  trip_miles  1.330983
2       tolls  1.293493
3        tips  1.071352


### Step 7 —  Fit on TRAIN Only, Predict on TEST, Evaluate MAE/RMSE

- Add intercept (`sm.add_constant`).
- Report **MAE/RMSE** in **units of Y**.
- Also capture **Adjusted R²** from the TRAIN fit summary to comment on fit (don’t use it alone for selection).


In [21]:
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

model = sm.OLS(y_train, X_train_const).fit()
predictions = model.predict(X_test_const)

# Model performance
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Adjusted R² (Train): {model.rsquared_adj:.3f}")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")

display(model.summary())


Adjusted R² (Train): 0.837
Mean Absolute Error (MAE): $3.66
Root Mean Squared Error (RMSE): $6.05


<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                            
===============================================================================
Dep. Variable:     base_passenger_fare   R-squared:                       0.837
Model:                             OLS   Adj. R-squared:                  0.837
Method:                  Least Squares   F-statistic:                 8.413e+06
Date:                 Tue, 11 Nov 2025   Prob (F-statistic):               0.00
Time:                         17:19:03   Log-Likelihood:            -2.1087e+07
No. Observations:              6552435   AIC:                         4.217e+07
Df Residuals:                  6552430   BIC:                         4.217e+07
Df Model:                            4                                         
Covariance Type:             nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 4.1870      0.005    929.508      0.000       4.178       4.196
trip_miles            1.6471      0.001   1735.128      0.000       1.645       1.649
trip_time_minutes     0.5239      0.000   1524.367      0.000       0.523       0.525
tolls                 0.3017      0.001    378.135      0.000       0.300       0.303
tips                  0.5161      0.001    557.281      0.000       0.514       0.518
==============================================================================
Omnibus:                  3514095.505   Durbin-Watson:                   1.999
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         52871301.210
Skew:                           2.249   Prob(JB):                         0.00
Kurtosis:                      16.169   Cond. No.                         43.1
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Step 8 —  Interpret Coefficients (Plain Language)
Write **unit-based** sentences “**holding others constant**.” Example templates (edit with your β values/units):

- **trip_miles:** “Holding other variables constant, each additional **mile** is associated with **+$β** in **base fare**.”
- **trip_time_minutes:** “Holding others constant, each additional **minute** is associated with **+$β** in **base fare**.”
- **tolls / tips:** interpret as “per $1 change,” holding others constant.

Also note **p-values** and whether they support including each predictor.

trip_miles (β = 1.6471):

Holding all other variables constant, each additional mile traveled is associated with an increase of $1.65 in the base passenger fare.
p-value: 0.000  Very significant; supports including this predictor.

trip_time_minutes (β = 0.5239):

Holding all other variables constant, each additional minute of trip time is associated with an increase of $0.52 in the base passenger fare.
p-value: 0.000  Very significant; supports including this predictor.

tolls (β = 0.3017):

Holding all other variables constant, each additional $1 of tolls is associated with an increase of $0.30 in the base passenger fare.
p-value: 0.000  Very significant; supports including this predictor.

tips (β = 0.5161):

Holding all other variables constant, each additional $1 of tips is associated with an increase of $0.52 in the base passenger fare.
p-value: 0.000  Very significant; supports including this predictor.

## We Share — Reflection & Wrap‑Up

Write **2 short paragraphs** and be specific:

1) **What changes did you make to handle multicollinearity and why?**  
Reference **corr**/**VIF** on TRAIN and any features you dropped or kept (with business rationale). Include **Adjusted R² (TRAIN)** and **TEST MAE/RMSE**.

2) **Stakeholder summary (units, one sentence):**  
Give a plain-English takeaway: e.g., “On unseen July trips, our typical error is about **$X** per fare; each extra mile adds about **$β_mile**, holding other factors constant.”


1. 
To address multicollinearity, first I noticed that trip_miles and trip_time_minutes were highly correlated, so I decided to drop trip_time_minutes to reduce redundancy while keeping trip_miles because it has clear business relevance, longer trips drive higher fares. Other predictors like tolls and tips had low correlation with each other and acceptable VIFs, so they were left alone. After adjustmenting, the Adjusted R² on train set remained high (0.837), indicating the model still explains a large portion of fare variability. On the test set, the model achieved an MAE of $3.66 and an RMSE of $6.05, confirming good predictive performance without multicollinearity issues.

2. 
On unseen trips, typical prediction error is about $3.66 per fare. Holding other factors constant, each additional mile adds roughly $1.65 to the base fare, each $1 in tolls adds about $0.30, and each $1 in tips adds about $0.52. This gives us actionable insight for fare forecasting and operational planning.